# Hossain Group HR Turnover Analytics BD

**A Kaggle-ready walkthrough using 762 fully synthetic Bangladesh-focused employee records.**

This notebook demonstrates how the dataset can be used to:

- inspect and validate HR records;
- calculate monthly headcount, hires, exits, and turnover;
- identify high-risk departments;
- analyse exit reasons;
- translate results into practical HR actions.

> **Important:** Hossain Group is a fictional project company. No real employee or confidential organisational data is included.


## 1. Import libraries and load the dataset

The notebook searches Kaggle's `/kaggle/input` directory first. When run from the GitHub repository, it falls back to the local project path.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams["figure.figsize"] = (12, 5)

candidates = list(Path("/kaggle/input").rglob("employee_master.csv"))
DATA_PATH = candidates[0] if candidates else Path("../data/raw/employee_master.csv")

print(f"Using dataset: {DATA_PATH}")

df = pd.read_csv(
    DATA_PATH,
    parse_dates=["Join_Date", "Exit_Date"],
)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")
print("\nFirst five records:")
print(df.head().to_string(index=False))


## 2. Data-quality snapshot

Before calculating turnover, confirm that the core HR fields are present and the dataset does not contain duplicate employee IDs or impossible date sequences.


In [ ]:
required_columns = {
    "Employee_ID",
    "Employee_Name",
    "Department",
    "Designation",
    "Location",
    "Employment_Type",
    "Join_Date",
    "Exit_Date",
    "Exit_Reason",
    "Status",
}

missing_columns = sorted(required_columns.difference(df.columns))
duplicate_ids = int(df["Employee_ID"].duplicated().sum())
invalid_exit_dates = int(
    (
        df["Exit_Date"].notna()
        & (df["Exit_Date"] < df["Join_Date"])
    ).sum()
)

quality_summary = pd.Series(
    {
        "Missing required columns": len(missing_columns),
        "Duplicate employee IDs": duplicate_ids,
        "Exit date before join date": invalid_exit_dates,
        "Missing department": int(df["Department"].isna().sum()),
        "Missing join date": int(df["Join_Date"].isna().sum()),
    },
    name="Data quality checks",
)

print(quality_summary.to_string())
if missing_columns:
    print("\nMissing columns:", missing_columns)


## 3. Monthly workforce metrics

The analysis covers **1 January 2025 to 30 June 2026**.

```text
Average Headcount = (Opening Headcount + Closing Headcount) / 2
Monthly Turnover Rate = Exits / Average Headcount
```


In [ ]:
analysis_start = pd.Timestamp("2025-01-01")
analysis_end = pd.Timestamp("2026-06-30")
months = pd.date_range(analysis_start, analysis_end, freq="MS")

def active_on(date):
    return (
        (df["Join_Date"] <= date)
        & (
            df["Exit_Date"].isna()
            | (df["Exit_Date"] >= date)
        )
    )

monthly_rows = []

for month_start in months:
    month_end = month_start + pd.offsets.MonthEnd(0)

    opening = int(active_on(month_start).sum())
    closing = int(
        (
            (df["Join_Date"] <= month_end)
            & (
                df["Exit_Date"].isna()
                | (df["Exit_Date"] > month_end)
            )
        ).sum()
    )
    hires = int(df["Join_Date"].between(month_start, month_end).sum())
    exits = int(df["Exit_Date"].between(month_start, month_end).sum())
    average_headcount = (opening + closing) / 2
    turnover_rate = exits / average_headcount if average_headcount else 0.0

    monthly_rows.append(
        {
            "Month": month_start,
            "OpeningHC": opening,
            "Hires": hires,
            "Exits": exits,
            "ClosingHC": closing,
            "AverageHC": average_headcount,
            "TurnoverRate": turnover_rate,
        }
    )

monthly = pd.DataFrame(monthly_rows)

print(monthly.head(6).to_string(index=False))


## 4. Executive KPI summary


In [ ]:
opening_headcount = int(monthly.iloc[0]["OpeningHC"])
closing_headcount = int(monthly.iloc[-1]["ClosingHC"])
total_hires = int(monthly["Hires"].sum())
total_exits = int(monthly["Exits"].sum())
average_headcount = float(monthly["AverageHC"].mean())
period_turnover = total_exits / average_headcount if average_headcount else 0.0
annualized_turnover = period_turnover * 12 / len(monthly)

def classify_risk(rate):
    if rate < 0.05:
        return "Low"
    if rate < 0.10:
        return "Moderate"
    if rate < 0.15:
        return "High"
    if rate < 0.20:
        return "Very High"
    return "Critical"

kpis = pd.Series(
    {
        "Analysis period": "Jan 2025 – Jun 2026",
        "Employee records": len(df),
        "Opening headcount": opening_headcount,
        "Closing headcount": closing_headcount,
        "Average headcount": round(average_headcount, 2),
        "Total hires": total_hires,
        "Total exits": total_exits,
        "Period turnover": f"{period_turnover:.2%}",
        "Annualized turnover": f"{annualized_turnover:.2%}",
        "Risk level": classify_risk(annualized_turnover),
    },
    name="Result",
)

print(kpis.to_string())


## 5. Monthly turnover trend


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    monthly["Month"],
    monthly["TurnoverRate"],
    marker="o",
    linewidth=2,
)
ax.set_title("Monthly Employee Turnover Rate")
ax.set_xlabel("Month")
ax.set_ylabel("Turnover Rate")
ax.yaxis.set_major_formatter(lambda value, position: f"{value:.1%}")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()

plt.show()


## 6. Department turnover exposure

Department risk should be assessed using exits relative to department headcount, not exits alone.


In [ ]:
department_rows = []

for department, department_df in df.groupby("Department"):
    department_monthly_average = []

    for month_start in months:
        month_end = month_start + pd.offsets.MonthEnd(0)

        opening = int(
            (
                (department_df["Join_Date"] <= month_start)
                & (
                    department_df["Exit_Date"].isna()
                    | (department_df["Exit_Date"] >= month_start)
                )
            ).sum()
        )
        closing = int(
            (
                (department_df["Join_Date"] <= month_end)
                & (
                    department_df["Exit_Date"].isna()
                    | (department_df["Exit_Date"] > month_end)
                )
            ).sum()
        )
        department_monthly_average.append((opening + closing) / 2)

    exits = int(
        department_df["Exit_Date"]
        .between(analysis_start, analysis_end)
        .sum()
    )
    avg_headcount = (
        sum(department_monthly_average) / len(department_monthly_average)
        if department_monthly_average
        else 0.0
    )
    annualized_rate = (
        exits / avg_headcount * 12 / len(months)
        if avg_headcount
        else 0.0
    )

    department_rows.append(
        {
            "Department": department,
            "Exits": exits,
            "AverageHC": avg_headcount,
            "AnnualizedTurnover": annualized_rate,
            "Risk": classify_risk(annualized_rate),
        }
    )

department_summary = (
    pd.DataFrame(department_rows)
    .sort_values("AnnualizedTurnover", ascending=False)
    .reset_index(drop=True)
)

printable_department_summary = department_summary.copy()
printable_department_summary["AverageHC"] = printable_department_summary["AverageHC"].round(2)
printable_department_summary["AnnualizedTurnover"] = printable_department_summary["AnnualizedTurnover"].map(
    lambda value: f"{value:.2%}"
)

print(printable_department_summary.to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(
    department_summary["Department"],
    department_summary["AnnualizedTurnover"],
)
ax.axhline(
    0.20,
    linestyle="--",
    linewidth=1.5,
    label="Critical threshold (20%)",
)
ax.set_title("Annualized Turnover by Department")
ax.set_xlabel("Department")
ax.set_ylabel("Annualized Turnover")
ax.yaxis.set_major_formatter(lambda value, position: f"{value:.0%}")
ax.tick_params(axis="x", rotation=45)
ax.legend()
fig.tight_layout()

plt.show()


## 7. Exit-reason analysis


In [ ]:
exited = df[df["Exit_Date"].between(analysis_start, analysis_end)].copy()

exit_reasons = (
    exited["Exit_Reason"]
    .fillna("Not Specified")
    .value_counts()
)

exit_reason_summary = pd.DataFrame(
    {
        "Exits": exit_reasons,
        "Share": exit_reasons / exit_reasons.sum(),
    }
)

printable_exit_reasons = exit_reason_summary.copy()
printable_exit_reasons["Share"] = printable_exit_reasons["Share"].map(
    lambda value: f"{value:.2%}"
)

print(printable_exit_reasons.to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.barh(
    exit_reasons.sort_values().index,
    exit_reasons.sort_values().to_numpy(),
)
ax.set_title("Employee Exits by Reason")
ax.set_xlabel("Employees")
ax.set_ylabel("Exit Reason")
fig.tight_layout()

plt.show()


## 8. HR interpretation

The analysis indicates that turnover should be treated as a workforce-risk signal requiring deeper diagnosis.

Recommended next steps:

1. prioritise departments with the highest annualized turnover;
2. separate voluntary, involuntary, and contract-completion exits;
3. compare exit reasons with tenure, designation, location, and employment type;
4. review compensation, career mobility, manager practices, workload, and work-life balance;
5. use stay interviews and targeted retention plans before applying broad interventions;
6. monitor monthly movement after each HR action.

Turnover should not be used as a standalone judgment of HR performance. It should be interpreted alongside hiring demand, workforce growth, contract patterns, business conditions, and employee feedback.


## 9. Responsible use

- The dataset is fully synthetic.
- Do not present it as authentic Hossain Group employee information.
- Do not use this notebook to make decisions about real employees without validated, authorised data and appropriate human review.
- Project source and citation:  
  https://github.com/samusa099/hossain-group-hr-turnover-analytics-bd
